In [1]:
!apt-get install zstd
!curl -fsSL https://ollama.com/install.sh | sh
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 124 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (704 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 125186 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%##################                 79.9%
>>> Creating ollama user...
>>> Adding ollama use

In [10]:
!pkill ollama
!pkill cloudflared

In [14]:
import subprocess
import time
import os

# 1. Matikan paksa ollama yang lama agar tidak bentrok
print("Membersihkan server lama...")
os.system("pkill ollama")
time.sleep(2)

# 2. Atur perizinan supaya terbuka untuk koneksi luar & Cloudflare
env = os.environ.copy()
env["OLLAMA_ORIGINS"] = "*"
env["OLLAMA_HOST"] = "0.0.0.0"

# 3. Jalankan ulang Ollama
print("Menghidupkan ulang server Ollama dengan akses publik...")
log_file = open("ollama.log", "w")
process = subprocess.Popen(["ollama", "serve"], env=env, stdout=log_file, stderr=log_file)
time.sleep(3)

print("✅ Selesai! Server Ollama sudah berjalan.")

Membersihkan server lama...
Menghidupkan ulang server Ollama dengan akses publik...
✅ Selesai! Server Ollama sudah berjalan.


In [15]:
import subprocess
import time
import re

print("🚀 Menjalankan Cloudflare Tunnel untuk Ollama (Port 11434)...")
log_file = open("cloudflared.log", "w")

# Menjalankan tunnel di background dan mengarahkannya ke localhost:11434
process = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://127.0.0.1:11434"], 
    stdout=log_file, 
    stderr=log_file
)

# Tunggu sekitar 8 detik agar Cloudflare selesai men-generate URL public
print("⏳ Menunggu URL dari Cloudflare...")
time.sleep(8)

# Membaca file log untuk mengekstrak URL
with open("cloudflared.log", "r") as f:
    log_content = f.read()

# Mencari link yang berakhiran .trycloudflare.com
url_match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log_content)

if url_match:
    print("\n✅ BERHASIL! Ollama Anda bisa diakses dari luar melalui URL berikut:")
    print(f"🔗 {url_match.group(0)}")
    print("\n(Copy URL di atas dan gunakan sebagai 'base_url' di aplikasi AI Anda)")
else:
    print("\n⚠️ URL belum muncul di log. Silakan tunggu beberapa detik lagi, lalu buka file 'cloudflared.log' secara manual untuk mengecek URL-nya.")

🚀 Menjalankan Cloudflare Tunnel untuk Ollama (Port 11434)...
⏳ Menunggu URL dari Cloudflare...

✅ BERHASIL! Ollama Anda bisa diakses dari luar melalui URL berikut:
🔗 https://jack-permission-plots-autos.trycloudflare.com

(Copy URL di atas dan gunakan sebagai 'base_url' di aplikasi AI Anda)


In [16]:
!wget -L "https://huggingface.co/theLittleStone/Qwen3.6-27B-AEON-Ultimate-Uncensored-MTP-GGUF/resolve/main/Qwen3.6-27B-AEON-Ultimate-Uncensored-BF16-MTP.Q4_K_M.gguf" -O model.gguf
!echo "FROM ./model.gguf" > Modelfile
!ollama create qwen-uncensored -f Modelfile

--2026-09-22 13:25:05--  https://huggingface.co/theLittleStone/Qwen3.6-27B-AEON-Ultimate-Uncensored-MTP-GGUF/resolve/main/Qwen3.6-27B-AEON-Ultimate-Uncensored-BF16-MTP.Q4_K_M.gguf
Resolving huggingface.co (huggingface.co)... 13.226.251.81, 13.226.251.66, 13.226.251.112, ...
Connecting to huggingface.co (huggingface.co)|13.226.251.81|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/6a4962fd8a91d410b5617af1/74709bc21cb55ab2cc7a1032bfb872e30d6fa8c2711f9842d40410252aec04e8?user_id=public&X-Xet-Cas-Uid=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27Qwen3.6-27B-AEON-Ultimate-Uncensored-BF16-MTP.Q4_K_M.gguf%3B+filename%3D%22Qwen3.6-27B-AEON-Ultimate-Uncensored-BF16-MTP.Q4_K_M.gguf%22%3B&Expires=1790087105&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNmE0OTYyZmQ4YTkxZDQxMGI1NjE3YWYxLzc0NzA5YmMyMWNiNTVhYjJjYzdhMTAzMmJmYjg3MmUzMGQ2ZmE4YzI3MTFmOTg0MmQ0MDQxMD

Testing Ollama

In [18]:
!ollama run qwen-uncensored "Halo, tes apakah kamu berfungsi?"

]11;?\⠙ ⠹ ⠸ ⠼ ⠴ ⠦ ⠧ ⠇ ⠏ ⠋ ⠙ ⠹ ⠸ ⠼ ⠴ ⠦ ⠧ ⠇ ⠏ ⠋ ⠙ ⠹ ⠸ ⠸ ⠼ ⠦ ⠦ ⠧ ⠇ ⠋ ⠋ ⠹ ⠹ ⠼ ⠼ ⠴ ⠧ ⠧ ⠏ ⠋ ⠙ ⠙ ⠹ ⠼ ⠼ ⠴ ⠦ ⠧ ⠏ ⠋ ⠋ ⠹ ⠹ ⠸ ⠼ ⠦ ⠦ ⠧ ⠇ ⠏ ⠋ ⠙ ⠸ ⠸ ⠴ ⠴ ⠧ ⠇ ⠏ ⠏ ⠙ ⠹ ⠹ ⠸ ⠴ ⠦ ⠦ ⠇ ⠏ ⠏ ⠋ ⠹ ⠸ ⠼ ⠴ ⠴ ⠦ ⠧ ⠇ ⠋ ⠙ ⠹ ⠸ ⠼ ⠴ ⠴ ⠦ ⠧ ⠇ ⠋ ⠙ ⠹ ⠸ ⠼ ⠼ ⠦ ⠧ ⠧ ⠇ ⠏ ⠋ ⠙ ⠸ ⠸ ⠴ ⠦ ⠧ ⠇ ⠇ ⠏ ⠋ ⠹ ⠹ ⠼ ⠴ ⠦ ⠦ ⠧ ⠇ ⠏ ⠙ ⠙ ⠹ ⠸ ⠼ ⠦ ⠧ ⠧ ⠏ ⠏ ⠙ ⠙ ⠹ ⠼ ⠴ ⠦ ⠧ ⠇ ⠇ ⠏ ⠙ ⠹ ⠸ ⠸ ⠴ ⠦ ⠦ ⠧ ⠇ ⠏ ⠋ ⠹ ⠸ ⠼ ⠼ ⠦ ⠧ ⠇ ⠏ ⠋ ⠙ ⠙ ⠸ ⠼ ⠴ ⠦ ⠧ ⠇ ⠇ ⠏ ⠋ ⠙ ⠹ ⠸ ⠼ ⠴ ⠧ ⠧ ⠇ ⠏ ⠋ ⠙ ⠸ ⠸ ⠴ ⠦ ⠦ ⠧ ⠇ ⠏ ⠋ ⠹ ⠹ ⠸ ⠴ ⠴ ⠧ ⠇ ⠇ ⠋ ⠙ ⠹ ⠸ ⠼ ⠴ ⠦ ⠧ ⠧ ⠏ ⠏ ⠙ ⠹ ⠸ ⠼ ⠴ ⠦ ⠦ ⠧ ⠇ ⠋ ⠋ ⠹ ⠸ ⠼ ⠴ ⠴ ⠧ ⠇ ⠏ ⠋ ⠙ ⠹ ⠸ ⠼ ⠼ ⠴ ⠧ ⠧ ⠏ ⠏ ⠋ ⠙ ⠹ ⠼ ⠼ ⠴ ⠦ ⠇ ⠏ ⠏ ⠋ ⠙ ⠹ ⠸ ⠼ ⠴ ⠧ ⠇ ⠏ ⠋ ⠋ ⠙ ⠹ ⠸ ⠼ ⠦ ⠦ ⠧ ⠇ ⠋ ⠋ ⠹ ⠸ ⠸ ⠼ ⠦ ⠦ ⠇ ⠏ ⠋ ⠋ ⠙ ⠹ ⠸ ⠼ ⠴ ⠧ ⠧ ⠏ ⠏ ⠋ ⠹ ⠹ ⠸ ⠼ ⠦ ⠦ ⠇ ⠇ ⠏ ⠙ ⠙ ⠸ ⠼ ⠼ ⠴ ⠦ ⠇ ⠏ Here's a thinking process:

1.  **Analyze User Input:**
   - **Language:** Indonesian ("Halo, tes apakah kamu berfungsi?")
   - **Meaning:** "Hello, testing if you are functioning?"
   - **Intent:** Simple check/test to see if the AI is responsive and worki
working properly.

2.  **Identify Key Components Needed in Response:**
   - Acknowledge the gree

In [14]:
import requests

url = "http://localhost:11434/api/generate"
data = {
    "model": "qwen-uncensored",
    "prompt": "Halo, tes apakah kamu berfungsi? Ceritakan 1 fakta unik tentang AI.",
    "stream": False
}

response = requests.post(url, json=data)
print(response.json().get('response', 'Tidak ada respons'))

We need to answer in Indonesian. The user said: "Halo, tes apakah kamu berfungsi? Ceritakan 1 fakta unik tentang AI." which translates to "Hello, test if you are functioning? Tell me 1 unique fact about AI."

So we need to respond in Indonesian, confirm functionality, and share one unique fact about AI.

Let's think of a unique fact about AI. Maybe something like: AI can sometimes exhibit "hallucinations" where they generate plausible but incorrect information, or maybe that AI models can be trained on different data to show biases, or something about how AI can learn from scratch. But we need one unique fact.

Perhaps: "AI tidak selalu membutuhkan data berlabel untuk belajar; pembelajaran tak terawasi memungkinkan AI menemukan pola dari data mentang." But that's common.

Maybe something more specific: "Beberapa model AI besar dapat menunjukkan 'efek keajaiban' di mana kinerja mereka melonjak tiba-tiba setelah mencapai jumlah parameter tertentu." That's about emergent abilities.

Or: "

In [21]:
import requests

url = "http://localhost:11434/v1/chat/completions" # Endpoint kompatibel OpenAI
data = {
    "model": "qwen-uncensored",
    "messages": [
        {"role": "system", "content": "Kamu adalah asisten AI yang cerdas dan asyik."},
        {"role": "user", "content": "Kenapa CPU mu naik? ga gpu aja"}
    ]
}

response = requests.post(url, json=data)
# Cara mengekstrak jawabannya juga mengikuti gaya JSON OpenAI
print(response.json()['choices'][0]['message']['content'])

Pertanyaan yang bagus! Banyak yang kira "lagi main game/edit video = GPU yang kerja keras", padahal CPU bisa naik tinggi juga tergantung kondisi. Ini alasannya:

🔹 **Peran berbeda**  
- **CPU**: Urus logika utama, perhitungan fisika, AI (misal: gerakan musuh, jalur NPC), *draw calls*, dan proses latar belakang.  
- **GPU**: Fokus rendering gambar, tekstur, pencahayaan, dan *post-processing*.  
Kedua komponen itu bergandeng tangan. Kalau salah satu "kenyang" duluan, yang lain bisa masih santai.

🔹 **Kenapa CPU lebih dulu naik?**
1. **Resolusi masih rendah** (1080p/1440p): GPU belum terbebani maksimal, jadi beban bergeser ke CPU. Di 4K biasanya GPU yang dominasi.
2. **Jenis game/aplikasi**: Game strategi, open-world, simulasi, atau game lama yang belum di-*shader-cache* seringkali sangat bergantung pada CPU.
3. **Pengaturan di dalam game**: Fitur seperti *Shadow Quality*, *AI Detail*, *Draw Distance*, atau *Physics* yang diset ke High/Ultra lebih makan CPU.
4. **Refresh rate tinggi**: La